# PlantDex Prototype (simple version)
Point at any plant and take a photo. The model gives the plant name. Wikipedia gives the details.

**Rules of this version**
1. The **model only gives the name** of the plant. No guessing of categories.
2. Details come from the **Wikipedia API** when you are online.
3. Every answer is saved in a **local cache file**. Next time the same plant shows up, we read the cache first.
4. If you are offline and the plant is not in the cache, we say so. No fake information.
5. Low confidence means we do not save the plant. We ask for a better photo.

Run the cells from top to bottom.

## Step 1: Install and import

In [ ]:
!pip -q install kagglehub pillow requests
import os, json, re, time, random, datetime, urllib.request
import numpy as np
import requests
import tensorflow as tf
from PIL import Image, ImageOps
print("TensorFlow:", tf.__version__)

## Step 2: Where to save your data
Colab deletes files when the session ends. If you want to keep your collection and cache, set `USE_DRIVE = True`. It saves in your Google Drive.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/plantdex"
else:
    DATA_DIR = "plantdex_data"
os.makedirs(DATA_DIR, exist_ok=True)

CACHE_FILE = os.path.join(DATA_DIR, "plant_cache.json")           # plant details from Wikipedia
COLLECTION_FILE = os.path.join(DATA_DIR, "collection.json")       # plants you caught
print("Saving to:", DATA_DIR)

## Step 3: Load the AIY Plants model

In [ ]:
MODEL_PATH = "plants_v1.tflite"
DIRECT_URL = "https://tfhub.dev/google/lite-model/aiy/vision/classifier/plants_V1/3?lite-format=tflite"

def try_direct():
    urllib.request.urlretrieve(DIRECT_URL, MODEL_PATH)
    tf.lite.Interpreter(model_path=MODEL_PATH)

def try_kaggle():
    import kagglehub, glob, shutil
    # If this fails, open the model page on Kaggle and copy the exact handle.
    folder = kagglehub.model_download("google/aiy/tfLite/vision-classifier-plants-v1")
    f = glob.glob(os.path.join(folder, "**", "*.tflite"), recursive=True)[0]
    shutil.copy(f, MODEL_PATH)

if not os.path.exists(MODEL_PATH):
    try:
        try_direct(); print("Loaded from direct link")
    except Exception as e:
        print("Direct link failed:", e)
        try_kaggle(); print("Loaded from Kaggle")

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]
print("Input:", inp["shape"], inp["dtype"], "| Output:", out["shape"])
print("Model size (MB):", round(os.path.getsize(MODEL_PATH) / 1e6, 1))

## Step 4: Load plant names

In [ ]:
import pandas as pd
LABEL_URL = "https://www.gstatic.com/aihub/tfhub/labelmaps/aiy_plants_V1_labelmap.csv"
LABELS = pd.read_csv(LABEL_URL).iloc[:, -1].astype(str).tolist()
BG_INDEX = [i for i, n in enumerate(LABELS) if n.lower() == "background"]
print("Labels:", len(LABELS), "| Examples:", LABELS[1:6])

## Step 5: Predict the plant name
To get a steadier answer, we look at the photo three ways (full photo, centre crop, flipped crop) and average the results.

In [ ]:
def run_model(pil_img):
    h, w = inp["shape"][1], inp["shape"][2]
    x = np.expand_dims(np.array(pil_img.resize((w, h))), 0)
    x = x.astype(np.uint8) if inp["dtype"] == np.uint8 else x.astype(np.float32) / 255.0
    interpreter.set_tensor(inp["index"], x)
    interpreter.invoke()
    y = interpreter.get_tensor(out["index"])[0].astype(np.float32)
    if out["dtype"] == np.uint8:
        scale, zero = out["quantization"]
        y = (y - zero) * scale if scale else y / 255.0
    return y

def classify(img, top_k=3):
    img = ImageOps.exif_transpose(img).convert("RGB")
    w, h = img.size
    s = min(w, h)
    center = img.crop(((w - s) // 2, (h - s) // 2, (w + s) // 2, (h + s) // 2))
    probs = np.mean([run_model(v) for v in (img, center, ImageOps.mirror(center))], axis=0)
    for i in BG_INDEX:
        if i < len(probs): probs[i] = 0
    n = min(len(probs), len(LABELS))
    best = np.argsort(probs[:n])[::-1][:top_k]
    return [(LABELS[i], float(probs[i])) for i in best]

## Step 6: Plant details from Wikipedia, with a local cache
Order of work:
1. Look in the cache file. If found, use it. No internet needed.
2. If not found and the device is online, ask Wikipedia and save the answer in the cache.
3. If not found and the device is offline, say so. We never invent details.

In [ ]:
HEADERS = {"User-Agent": "PlantDexStudentProject/1.0 (college learning project)"}
PLANT_WORDS = ["plant", "species", "genus", "tree", "flower", "shrub", "herb", "grass", "family", "vine", "fern"]

CACHE = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}

def save_cache():
    json.dump(CACHE, open(CACHE_FILE, "w"), indent=1)

def is_online():
    try:
        requests.get("https://en.wikipedia.org", headers=HEADERS, timeout=4)
        return True
    except Exception:
        return False

def wiki_page(title):
    url = "https://en.wikipedia.org/api/rest_v1/page/summary/" + requests.utils.quote(title.replace(" ", "_"), safe="")
    r = requests.get(url, headers=HEADERS, timeout=10)
    if r.status_code != 200:
        return None
    d = r.json()
    text = (d.get("extract") or "").lower()
    if d.get("type") != "standard" or not any(w in text for w in PLANT_WORDS):
        return None
    return d

def fetch_from_wikipedia(name):
    d = wiki_page(name)
    if d is None:  # search, then open the best match
        r = requests.get("https://en.wikipedia.org/w/api.php", headers=HEADERS, timeout=10,
                         params={"action": "query", "list": "search", "srsearch": name + " plant",
                                 "format": "json", "srlimit": 1})
        hits = r.json().get("query", {}).get("search", [])
        if hits:
            d = wiki_page(hits[0]["title"])
    if d is None:
        return None
    sentences = re.split(r"(?<=[.!?])\s+", d["extract"].strip())
    return {"title": d.get("title"), "short": d.get("description"),
            "summary": " ".join(sentences[:3]),
            "image": (d.get("thumbnail") or {}).get("source"),
            "url": d.get("content_urls", {}).get("desktop", {}).get("page"),
            "source": "Wikipedia", "saved_on": str(datetime.date.today())}

def get_details(name):
    """Returns (details or None, where it came from)."""
    if name in CACHE:
        return CACHE[name], "cache"
    if not is_online():
        return None, "offline"
    try:
        info = fetch_from_wikipedia(name)
    except Exception as e:
        print("Wikipedia problem:", e)
        return None, "error"
    if info is None:
        return None, "not found"
    CACHE[name] = info
    save_cache()
    return info, "wikipedia"

def show_details(name):
    info, src = get_details(name)
    if info is None:
        msgs = {"offline": "You are offline and this plant is not saved yet. Connect to the internet and try again.",
                "not found": "No Wikipedia page found for this plant.",
                "error": "Could not get details right now. Try again later."}
        print(msgs[src]); return
    print(f"Name: {info['title']}" + (f" ({info['short']})" if info.get("short") else ""))
    print(f"About: {info['summary']}")
    if info.get("url"): print(f"Read more: {info['url']}")
    print(f"[from {'saved cache' if src == 'cache' else 'Wikipedia, now saved to cache'}]")

## Step 7: Collection and levels
- Each new plant gives 10 XP
- Level goes up every 3 plants
- A plant is saved only if the model is confident enough

In [ ]:
CONFIDENT = 0.50   # at or above this: caught
MAYBE = 0.25       # between MAYBE and CONFIDENT: ask for a better photo

def load_collection():
    return json.load(open(COLLECTION_FILE)) if os.path.exists(COLLECTION_FILE) else {"plants": {}, "xp": 0}

def save_collection(c):
    json.dump(c, open(COLLECTION_FILE, "w"), indent=2)

def catch(results):
    name, conf = results[0]
    if conf < MAYBE:
        print(f"Not sure what this is. Best guess: {name} ({conf:.0%}).")
        print("Tip: take a clear, close photo of one leaf or flower in good light.")
        return
    if conf < CONFIDENT:
        print(f"Maybe {name} ({conf:.0%}), but I am not sure enough to save it.")
        print("Tip: try again with a closer photo of one leaf or flower.")
        return
    c = load_collection()
    if name in c["plants"]:
        print(f"You already caught {name}.")
    else:
        c["plants"][name] = {"date": str(datetime.date.today()), "confidence": round(conf, 2)}
        c["xp"] += 10
        save_collection(c)
        print(f"GOTCHA! {name} ({conf:.0%}) added to your PlantDex. +10 XP")
    print()
    show_details(name)

def show_dex():
    c = load_collection()
    total = len(c["plants"])
    print(f"Plants caught: {total}   XP: {c['xp']}   Level: {total // 3 + 1}")
    print("-" * 40)
    for name, p in c["plants"].items():
        print(f"{name}  (caught {p['date']}, {p['confidence']:.0%})")

## Step 8: Daily quest

In [ ]:
QUESTS = [
    "Point at any plant and take a photo.",
    "Find any plant and take a clear photo of one leaf.",
    "Find any plant with a flower and take a close photo.",
    "Find any plant you have never looked at before and take a photo.",
    "Take a photo of any plant growing near your home.",
]

def todays_quest():
    return QUESTS[random.Random(datetime.date.today().toordinal()).randrange(len(QUESTS))]

print("Today's quest:", todays_quest())

## Step 9: Try it
Upload a plant photo. Best results: one plant, close, sharp, good light.

In [ ]:
from google.colab import files
from IPython.display import display

uploaded = files.upload()
for fname in uploaded:
    img = Image.open(fname)
    display(ImageOps.exif_transpose(img).convert("RGB").resize((250, 250)))
    results = classify(img)
    print("Model guesses:")
    for n, p in results:
        print(f"  {n}   {p:.0%}")
    print()
    catch(results)

## Step 10: See your PlantDex

In [ ]:
show_dex()

## Next steps
1. Move the model into a web app (TensorFlow.js or ONNX Runtime Web).
2. Keep the same cache idea in the browser with **IndexedDB**.
3. Deploy on **Render** as a PWA.
4. Test outside on real plants and write about how it went.

**Notes**
- The model knows about 2,000 plants. Plants outside that list will get wrong guesses. That is why we use confidence limits.
- Wikipedia text is free but needs credit. Always show the Read more link.
- Check the model license on its page and mention it in your post.